# Lab controls — every LD1 correlate with and without lab

Numbers from `lab_controls.py` (Pearson, as the paper's figures: newest `results/lab_controls_pearson_<date>.csv` and `results/lab_controls/ldvar.csv`); this notebook only draws them.

| panel | what |
|---|---|
| a | behavioural correlates of LD1 across mice: Pearson r (open) and the partial correlation with lab removed (filled), 95 % CI from mice resampled within lab; stars = BH q of the lab-removed test over the whole table |
| b | neural correlates: the fig_neural partial r (region + trial-count covariates), without and with lab dummies; CI from sessions resampled within lab; q as in a |
| c | lab share of each LD's between-mouse variance (η² on mouse means) against the chance level of the lab labels (shuffled across mice, lab sizes kept): line = null mean, band = up to its 95th percentile |

**Read with:** η² has a floor of (labs − 1) / (mice − 1) ≈ 0.18 with 11 labs and 56 mice, so the bars are judged against the band, not against zero.

In [ ]:
import sys, pathlib, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
for p in [str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
import paper_style as ps
import fig_layout as fl
ps.use('paper')
FIG = 'fig_lab_controls'

## Inputs and the rows drawn

`ROWS_A` / `ROWS_B`: (group, correlate as named in the table, label drawn). A correlate missing from the table (a section not run on this machine) is skipped; the time-horizon rows appear once `lab_controls.py --only horizon` has run.

In [ ]:
TABLE = sorted(glob.glob(str(PI / 'results' / 'lab_controls_pearson_*.csv')),
               key=lambda f: pd.to_datetime(re.search(r'(\d\d-\d\d-\d{4})', f).group(1), dayfirst=True))[-1]
T = pd.read_csv(TABLE)
T = T[T['section'] != 'neural_mouse'].set_index('correlate')   # mouse-unit re-tests of the neural rows: same names
LDVAR = pd.read_csv(PI / 'results' / 'lab_controls' / 'ldvar.csv')
print(pathlib.Path(TABLE).name, len(T), 'rows')

ROWS_A = [
    ('Syllable use', 'Whisking (whole trial)', 'Whisking'),
    ('Syllable use', 'Whisking (quiescence)', 'Whisking (quiescence)'),
    ('Syllable use', 'Licking (whole trial)', 'Licking'),
    ('Syllable use', 'Idle paw state (0)', 'Idle paw state'),
    ('Syllable use', 'Mean paw-state vigor rank', 'Paw-state vigor'),
    ('Task performance', 'Reaction time (log)', 'Reaction time'),
    ('Task performance', 'Accuracy (easy trials)', 'Accuracy'),
    ('Task performance', 'Choice bias', 'Choice bias'),
    ('Task performance', 'Training time (log days)', 'Training time'),
    ('Internal states (GLM-HMM)', 'State switches (K=2)', 'State switches'),
    ('Internal states (GLM-HMM)', 'State switches | easy accuracy', 'Switches | accuracy'),
    ('Internal states (GLM-HMM)', 'State switches (K=4)', 'Switches (K = 4)'),
    ('Internal states (GLM-HMM)', 'Short disengaged bouts (<=2 trials)', 'Brief lapses (≤ 2 trials)'),
    ('Internal states (GLM-HMM)', 'P(stay engaged)', 'P(stay engaged)'),
    ('Internal states (GLM-HMM)', 'P(stay disengaged)', 'P(stay disengaged)'),
    ('Internal states (GLM-HMM)', 'Posterior uncertainty', 'Posterior uncertainty'),
    ('Internal states (GLM-HMM)', 'Engaged log posterior ratio', 'Engaged posterior'),
    ('Internal states (GLM-HMM)', 'Fraction engaged (hard)', 'Fraction engaged'),
    ('Stereotypy', 'Trial-mode entropy', 'Trial-mode entropy'),
] + [('Time horizon (vs Proficient)', c, c.replace(' vs Proficient', ''))
     for c in T.index[T['section'] == 'horizon']]
ROWS_B = [(m, f'{m}, {t}', lab) for m in ['Firing rate', 'Fano factor', 'Noise correlation']
          for t, lab in [('Baseline', 'Pre'), ('Post-stimulus', 'Post'), ('Response magnitude', 'Change')]]
missing = [c for _, c, _ in ROWS_A + ROWS_B if c not in T.index]
print('not in the table (skipped):', missing or 'none')

## Forest plot (a, b)

One row per correlate: open = as reported (across mice; for neural, the paper's test), filled = lab removed. Stars at the right: q of the lab-removed test. Group names sit on their own row above their correlates.

In [ ]:
RAW_C, LAB_C = '0.62', '0.12'
OFF = 0.17          # raw above, lab below the row's centre

def forest(ax, rows, xlabel):
    rows = [r for r in rows if r[1] in T.index]
    y, ticks, labels, heads, last = 0, [], [], [], None
    for group, c, lab in rows:
        if group != last:
            heads.append((y, group)); y += 1; last = group
        r = T.loc[c]
        for v, lo, hi, col, dy, filled in [(r['rho'], r['rho_lo'], r['rho_hi'], RAW_C, -OFF, False),
                                           (r['rho_lab'], r['rho_lab_lo'], r['rho_lab_hi'], LAB_C, OFF, True)]:
            ax.plot([lo, hi], [y + dy] * 2, color=col, lw=0.8, solid_capstyle='butt')
            ax.plot(v, y + dy, 'o', ms=3.2, color=col, mfc=col if filled else 'white', mew=0.8, zorder=3)
        if r['q_lab'] < 0.05:
            ax.text(1.0, y, ps.stars(r['q_lab']), transform=blended_transform_factory(ax.transAxes, ax.transData),
                    ha='left', va='center', color=LAB_C, fontsize=plt.rcParams['font.size'] * 0.85)
        ticks.append(y); labels.append(lab); y += 1
    ax.axvline(0, color='0.5', lw=0.5, zorder=0)
    ax.set_yticks(ticks); ax.set_yticklabels(labels)
    ax.tick_params(axis='y', length=0)
    ax.set_ylim(y - 0.4, -0.7)
    tr = blended_transform_factory(ax.transAxes, ax.transData)
    for hy, g in heads:
        ax.text(-0.02, hy, g, transform=tr, ha='right', va='center', fontweight='bold',
                fontsize=plt.rcParams['font.size'] * 0.9)
    ax.spines['left'].set_visible(False)
    ax.set_xlabel(xlabel)
    return ax

LEGEND = [Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.2, color=RAW_C, mfc='white', label='as reported'),
          Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.2, color=LAB_C, label='lab removed')]

def draw_A(fig, spec):
    ax = fig.add_subplot(spec)
    forest(ax, ROWS_A, 'Pearson r with LD1 (across mice)')
    ax.set_xlim(-0.85, 0.85)
    ax.legend(handles=LEGEND, frameon=False, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, handlelength=1.6,
              borderaxespad=0.1)
    return [ax]

def draw_B(fig, spec):
    ax = fig.add_subplot(spec)
    forest(ax, ROWS_B, 'Partial correlation with LD1')
    ax.set_xlim(-0.4, 0.4)
    return [ax]

fig = plt.figure(figsize=(3.2, 4.2)); draw_A(fig, fig.add_gridspec(1, 1, left=0.45)[0]); plt.show()
fig = plt.figure(figsize=(3.2, 2.4)); draw_B(fig, fig.add_gridspec(1, 1, left=0.45)[0]); plt.show()

## Lab variance per LD (c)

Bars: η² of lab on the mouse means of each LD. Line and band: the same statistic with lab labels shuffled across mice (lab sizes kept) — mean and 95th percentile. Stars: permutation p.

In [ ]:
def draw_C(fig, spec):
    ax = fig.add_subplot(spec)
    d = LDVAR
    x = d['LD'].to_numpy()
    ax.fill_between(np.r_[x - 0.5, x[-1] + 0.5], 0, np.r_[d['null_95'], d['null_95'].iloc[-1]], step='post',
                    color=ps.NULL_BAND, alpha=0.25, lw=0, label='shuffled labs (95%)')
    ax.step(np.r_[x - 0.5, x[-1] + 0.5], np.r_[d['null_mean'], d['null_mean'].iloc[-1]], where='post',
            color='0.45', lw=0.7, ls='--', label='shuffled labs (mean)')
    ax.bar(x, d['eta2_lab'], width=0.62, color=[LAB_C if p < 0.05 else 'white' for p in d['p']], edgecolor=LAB_C, lw=0.6)
    for xi, v, p in zip(x, d['eta2_lab'], d['p']):
        if p < 0.05:
            ax.text(xi, v + 0.015, ps.stars(p), ha='center', va='bottom', fontsize=plt.rcParams['font.size'] * 0.8)
    ax.set_xticks(x); ax.set_xticklabels([f'{k}' for k in x])
    ax.set_xlim(0.4, x[-1] + 0.6); ax.set_ylim(0, 0.7); ax.set_yticks([0, 0.3, 0.6])
    ax.set_xlabel('Linear discriminant'); ax.set_ylabel('Lab share of\nbetween-mouse variance')
    ax.legend(frameon=False, loc='upper right', handlelength=1.4, borderaxespad=0.1)
    return [ax]

fig = plt.figure(figsize=(3.2, 1.8)); draw_C(fig, fig.add_gridspec(1, 1, left=0.25, bottom=0.25)[0]); plt.show()

## LD1's syllable loadings with and without lab (d)

`fig_ld1` panel B at the mouse level: per (time bin, syllable), the Pearson correlation of mouse-mean occupancy with mouse-mean LD1 (top), and the partial correlation with lab removed (bottom). A qualitative description, read as in fig_ld1 B: grey band = each cell's permutation null (2.5–97.5 %; LD1 shuffled across mice on top, within lab below), drawn as the widest band over the features shown; a line is drawn bold where it leaves its own null, faint inside it. No correction across cells. From `results/lab_controls/loading_map.csv`.

In [ ]:
import matplotlib.patheffects as pe
ps.use_paw_states('uniform')
LMAP = pd.read_csv(PI / 'results' / 'lab_controls_pearson' / 'loading_map.csv')
N_BINS = LMAP['bin'].nunique()
FEAT_LABEL = [ps.paw_label(k) for k in range(ps.N_PAW_STATES)] + ['Whisk', 'Lick']
OUTLINE = [pe.Stroke(linewidth=plt.rcParams['lines.linewidth'] + 0.9, foreground='0.35'), pe.Normal()]

def loading_axes(ax_p, ax_w, r_col, null, title):
    x = np.arange(N_BINS)
    for ax, feats in [(ax_p, range(ps.N_PAW_STATES)), (ax_w, [8, 9])]:
        sub = LMAP[LMAP['syllable'].isin(list(feats))]
        lo = sub.groupby('bin')[f'null_{null}_lo'].min().to_numpy(); hi = sub.groupby('bin')[f'null_{null}_hi'].max().to_numpy()
        ax.fill_between(x, lo, hi, color=ps.NULL_BAND, alpha=0.3, lw=0)
        for f in feats:
            d = LMAP[LMAP['syllable'] == f].sort_values('bin')
            y = d[r_col].to_numpy()
            sig = ((y < d[f'null_{null}_lo']) | (y > d[f'null_{null}_hi'])).to_numpy()
            ax.plot(x, y, color=ps.SYLLABLE_COLORS[f], alpha=0.3, lw=0.8)
            ax.plot(x, np.where(sig, y, np.nan), color=ps.SYLLABLE_COLORS[f], path_effects=OUTLINE, label=FEAT_LABEL[f])
        ax.axhline(0, color=ps.NEUTRAL, lw=0.6)
        ps.epoch_lines(ax, N_BINS, label=ax_p is not None and title is not None, names=ps.EPOCH_SHORT)
        ax.set_xlim(0, N_BINS - 1); ax.set_xticks([]); ax.set_ylim(-0.8, 0.95)
    ax_w.tick_params(labelleft=False)
    ax_p.set_ylabel(title)

def draw_D(fig, spec):
    g = spec.subgridspec(2, 2, wspace=0.08, hspace=0.35)
    axes = []
    for i, (r_col, null, lab) in enumerate([('r_raw', 'raw', 'r with LD1\n(across mice)'),
                                              ('r_lab', 'lab', 'r with LD1\n(lab removed)')]):
        ax_p = fig.add_subplot(g[i, 0]); ax_w = fig.add_subplot(g[i, 1], sharey=ax_p)
        loading_axes(ax_p, ax_w, r_col, null, lab)
        if i == 0:
            ax_p.set_title('Paw state', pad=12); ax_w.set_title('Whisking / licking', pad=12)
        else:
            for ax in (ax_p, ax_w):
                for t in list(ax.texts):
                    t.remove()
        axes += [ax_p, ax_w]
    kw = dict(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.04), handlelength=1.2, columnspacing=0.8,
              fontsize=plt.rcParams['legend.fontsize'] * 0.9)
    axes[2].legend(ncol=3, **kw); axes[3].legend(ncol=1, **kw)
    return axes

fig = plt.figure(figsize=(4.6, 3.6)); draw_D(fig, fig.add_gridspec(1, 1, bottom=0.2)[0]); plt.show()

## The assembled figure

180 mm wide: a on the left (all behavioural rows), b over c on the right.

In [ ]:
fig = plt.figure(figsize=(fl.FIG_WIDTH, 5.4))
outer = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.95, left=0.24, right=0.97, top=0.93, bottom=0.09)
right = outer[1].subgridspec(2, 1, height_ratios=[1.25, 1], hspace=0.6)   # lower right: free (the lab share per LD is fig_lab_share)
panels = {'a': draw_A(fig, outer[0]), 'b': draw_B(fig, right[0])}
fl.place_letters(fig, panels, rows=['ab'], left='a')
plt.show()

OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf']:
    fig.savefig(path)
    print('wrote', path.relative_to(PI))

## COMPACT version for the main text

Every LD1 test the paper's figures show, significant or not (fig_ld1 C, fig_engagement, fig_engagement_syllables G and I, fig_neural). Each column is one way of computing the same correlation, and each is a test in its own right: a dot (the correlation), its 95 % CI, one hue darkening with the stringency of the unit (sessions or recordings → mice → mice within lab), and filled where that column's test has BH q < 0.05 over the paper family (open: n.s.).

* **a** behaviour. *Across sessions*: Pearson r over sessions, as the paper's panels draw it (CI: Fisher z; sessions of a mouse are not independent, so this column is optimistic). *Across mice*: session values averaged within mouse. *Lab removed*: partial correlation across mice with lab regressed out of both (CI: mice resampled within lab; p: LD1 permuted within lab). Training time (*) is one value per mouse, so it has no session column.
* **b** neural. *Across recordings*: fig_neural's partial r (region and trial-count covariates, LD1 permuted across sessions). *Lab removed*: lab dummies added, LD1 permuted within lab. CI: sessions resampled within lab.
(The lab share of each LD's between-mouse variance, formerly panel c here, is its own supplementary figure: `fig_lab_share.ipynb`.)

In [ ]:
# COMPACT
from matplotlib.patches import Patch
# one column per way of computing the correlation; each column is its own test, with its own CI and its own
# significance (filled: BH q < 0.05 over the paper family, separately per column)
COLS_BEH = [('rho_session', 'q_session', 'Across\nsessions'),
            ('rho', 'q', 'Across\nmice'),
            ('rho_lab', 'q_lab', 'Across mice,\nlab removed')]
COLS_NEU = [('rho', 'q', 'Across\nrecordings'),
            ('rho_lab', 'q_lab', 'Lab\nremoved')]
# one sequential hue, darker = stricter: sessions (and recordings, also session-level) -> mice -> mice within lab
COL_C = {'Across\nsessions': '#7FB0D8', 'Across\nrecordings': '#7FB0D8', 'Across\nmice': '#3A78B0',
         'Across mice,\nlab removed': '#12355B', 'Lab\nremoved': '#12355B'}
ROWS_COMPACT = [('Task', 'Accuracy (easy trials)', 'Accuracy'),
                ('Task', 'Choice bias', 'Choice bias'),
                ('Task', 'Block bias shift', 'Block bias shift'),
                ('Task', 'Reaction time (log)', 'Reaction time'),
                ('Task', 'Trial elongation (log)', 'Trial elongation'),
                ('Task', 'Training time (log days)', 'Training time*'),
                ('Internal states', 'Fraction engaged (hard)', 'Fraction engaged'),
                ('Internal states', 'Engaged log posterior ratio', 'Engaged posterior'),
                ('Internal states', 'State switches (K=2)', 'State switches'),
                ('Internal states', 'State switches | easy accuracy', 'Switches | accuracy'),
                ('Internal states', 'P(stay engaged)', 'P(stay engaged)'),
                ('Internal states', 'P(stay disengaged)', 'P(stay disengaged)'),
                ('Internal states', 'Held-out fit gain K=2->3', 'Fit gain, K = 2 → 3'),
                ('Internal states', 'Held-out fit gain K=2->4', 'Fit gain, K = 2 → 4'),
                ('Internal states', 'State switches (K=4)', 'Switches, K = 4'),
                ('Entropy', 'Trial-mode entropy', 'Trial-mode entropy'),
                ('Entropy', 'Syllable entropy', 'Syllable entropy')]
ROWS_NEURAL = [(g, f'{m}, {t}', f'{short}, {lab}') for m, g, short in
               [('Firing rate', 'Firing rate', 'FR'), ('Fano factor', 'Fano factor', 'FF'), ('Noise correlation', 'Noise corr.', 'r$_{SC}$')]
               for t, lab in [('Baseline', 'pre'), ('Post-stimulus', 'post'), ('Response magnitude', 'change')]]

def columns(fig, spec, rows, cols, xlim, xticks, xlabel, labels=True):
    rows = [r for r in rows if r[1] in T.index]
    groups = pd.Series([r[0] for r in rows])
    ys = np.arange(len(rows))
    g = spec.subgridspec(1, len(cols), wspace=0.12)
    axes = []
    for j, (vcol, qcol, title) in enumerate(cols):
        ax = fig.add_subplot(g[j], sharey=axes[0] if axes else None)
        col = COL_C[title]
        for y, (_, c, _) in zip(ys, rows):
            r = T.loc[c]
            v = r.get(vcol, np.nan)
            if not np.isfinite(v):
                continue
            lo, hi = r.get(f'{vcol}_lo', np.nan), r.get(f'{vcol}_hi', np.nan)
            sig = r.get(qcol, np.nan) < 0.05
            ax.plot([lo, hi], [y, y], color=col, lw=0.8, solid_capstyle='butt')
            ax.plot(v, y, 'o', ms=3.4, color=col, mfc=col if sig else 'white', mew=0.8, zorder=3)
        ax.axvline(0, color='0.6', lw=0.5, zorder=0)
        for gname in dict.fromkeys(groups):                 # thin separators between groups
            i0 = np.flatnonzero(groups == gname).min()
            if i0 > 0:
                ax.axhline(i0 - 0.5, color='0.85', lw=0.5, zorder=0)
        ax.set_xlim(*xlim); ax.set_xticks(xticks)
        ax.set_title(title, color=col, fontsize=plt.rcParams['font.size'] * 0.9, pad=3)
        ax.tick_params(axis='y', length=0)
        ax.spines['left'].set_visible(False)
        if j > 0:
            ax.tick_params(labelleft=False)
        axes.append(ax)
    axes[0].set_yticks(ys); axes[0].set_yticklabels([r[2] for r in rows] if labels else [])
    axes[0].set_ylim(len(rows) - 0.4, -0.6)
    # group names left of the row labels
    tr = blended_transform_factory(fig.transFigure, axes[0].transData)
    axes[len(cols) // 2].set_xlabel(xlabel)
    return axes, groups

def group_labels(fig, ax, groups, x):
    tr = blended_transform_factory(fig.transFigure, ax.transData)
    for gname in dict.fromkeys(groups):
        idx = np.flatnonzero(groups == gname)
        ax.text(x, (idx.min() + idx.max()) / 2, gname, transform=tr, rotation=90, ha='center', va='center',
                color='0.4', fontsize=plt.rcParams['font.size'] * 0.85, clip_on=False)
        ax.plot([x + 0.008] * 2, [idx.min() - 0.3, idx.max() + 0.3], transform=tr, color='0.6', lw=0.6, clip_on=False)

def compact_a(fig, spec):
    axes, groups = columns(fig, spec, ROWS_COMPACT, COLS_BEH, (-0.8, 0.8), [-0.5, 0, 0.5], 'Pearson r with LD1')
    group_labels(fig, axes[0], groups, 0.012)
    return axes

def compact_b(fig, spec):
    axes, _ = columns(fig, spec, ROWS_NEURAL, COLS_NEU, (-0.35, 0.2), [-0.2, 0], 'Partial r with LD1')
    return axes

figc = plt.figure(figsize=(fl.FIG_WIDTH, 3.9))
gc = figc.add_gridspec(1, 2, width_ratios=[1.25, 0.85], wspace=0.55, left=0.17, right=0.985, top=0.9, bottom=0.1)
gr = gc[1].subgridspec(2, 1, height_ratios=[1.0, 0.85], hspace=0.75)      # b on top; below it free (lab share: fig_lab_share)
panels_c = {'a': compact_a(figc, gc[0]), 'b': compact_b(figc, gr[0])}
key = [Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.4, color='#3A78B0', label='q < 0.05'),
       Line2D([], [], marker='o', ls='-', lw=0.8, ms=3.4, color='#3A78B0', mfc='white', label='n.s.')]
figc.legend(handles=key, frameon=False, loc='upper left', bbox_to_anchor=(0.035, 0.995), ncol=2, handlelength=1.4,
            columnspacing=0.8)
fl.place_letters(figc, panels_c, rows=['ab'], left='a')
plt.show()
for path in [FIGDIR / 'figures' / f"{FIG}_compact_{date.today().strftime('%d-%m-%Y')}.svg", FIGDIR / 'figures' / f'{FIG}_compact.pdf',
             FIGDIR / 'figures' / f'{FIG}_compact.png']:
    figc.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))